# Chapter 26: Selected Topics in Hyperbolic Geometry

**Source orientation.** Sections 26.1-26.5; printed pages 505-524; PDF pages 527-546. The source span is used for structure and mathematical coverage only. The prose, code, diagrams, numeric examples, and checks below are original notebook material.

**Chapter question.** Which Euclidean-looking pictures change their meaning inside the hyperbolic disk?

**Goal.** By the end of this notebook, you should be able to distinguish hyperbolic cycles in the Poincare disk, read triangle area as angle defect, test the hyperbolic versions of Thales and Pythagoras, construct regular hyperbolic n-gons from angle data, and recognize symmetry-group orbits as metric objects rather than Euclidean-size pictures.

In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if candidate.name == "Perspectives-on-Projective-Geometry" and (candidate / "AGENTS.md").exists():
        BOOK_ROOT = candidate
        break
    nested = candidate / "Perspectives-on-Projective-Geometry"
    if (nested / "AGENTS.md").exists():
        BOOK_ROOT = nested
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Perspectives-on-Projective-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER_SLUG = "chapter-26-selected-topics-in-hyperbolic-geometry"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
for child in ("figures", "html", "tables", "checks"):
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

print("BOOK_ROOT", BOOK_ROOT.relative_to(BOOK_ROOT.parent))
print("ARTIFACT_ROOT", ARTIFACT_ROOT.relative_to(BOOK_ROOT))

## Computational Translation Guide

The chapter keeps returning to one idea: a picture in the disk may look Euclidean, but the measuring rule is hyperbolic.

| Book idea | Computational representation | What to inspect |
| --- | --- | --- |
| Proper circles, horocycles, hypercycles | Euclidean circles and inversion-related arcs inside the Poincare disk | Whether the center is inside, on, or outside the absolute, and whether one or two visible branches appear |
| A-circles versus C-circles | Four inversion choices for three points versus one Euclidean circle through them | Algebraic completion produces more cycles than the constant-curvature branch viewpoint |
| Area and angle defect | Poincare geodesic triangles with conformal Euclidean angle measurement | The defect `pi - angle_sum` grows as vertices approach the boundary |
| Hyperbolic Thales | A polarity construction in the Beltrami-Klein disk | The right-angle locus is a conic, while generic peripheral-angle loci are not so simple |
| Hyperbolic Pythagoras | Distance checks using `cosh(a) cosh(b) = cosh(c)` | The Euclidean additive form needs a hyperbolic correction term |
| Regular n-gons and tilings | Right-triangle decomposition with angles `pi/n` and `pi/m` | Existence is controlled by `1/n + 1/m < 1/2` |
| Symmetry groups | Orbits under disk-preserving Mobius transformations | Isometries preserve hyperbolic distances while Euclidean spacing collapses near the boundary |

**Library routing.** Matplotlib is used for durable 2D disk diagrams and proof sketches; Plotly is used for interactive polygon and orbit labs; SymPy is used for exact series and bilinear identity checks; NetworkX is used for the proof dependency graph; pandas records reproducible numeric ledgers.

In [ ]:
import itertools
import json
import math
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import artifact_path, assert_artifacts, book_relative, display_artifact, image_stats, save_json
from utils.cayley_klein import poincare_distance
from utils.projective import cross_ratio

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

artifact_registry = {}
check_data = {}
ABSOLUTE_CONIC = np.diag([1.0, 1.0, -1.0])


def rel(path: Path) -> str:
    return book_relative(path)


def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.ndarray):
        return clean_json(value.tolist())
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, Path):
        return rel(value)
    return value


def save_fig(fig, filename: str) -> Path:
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    artifact_registry[filename] = path
    return path


def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = artifact_path(ARTIFACT_ROOT, "tables", filename)
    df.to_csv(path, index=False)
    artifact_registry[filename] = path
    return path


def save_plotly(fig: go.Figure, filename: str) -> Path:
    path = artifact_path(ARTIFACT_ROOT, "html", filename)
    html = fig.to_html(include_plotlyjs="cdn", full_html=True)
    path.write_text(html, encoding="utf-8")
    artifact_registry[filename] = path
    return path


def disk_boundary(ax):
    t = np.linspace(0, 2 * np.pi, 500)
    ax.plot(np.cos(t), np.sin(t), color="black", lw=1.1)
    ax.fill(np.cos(t), np.sin(t), color="#f7f7f7", zorder=-5)
    ax.set_aspect("equal")
    ax.set_xlim(-1.08, 1.08)
    ax.set_ylim(-1.08, 1.08)
    ax.set_xticks([])
    ax.set_yticks([])


def hpoint_xy(p):
    p = np.asarray(p, dtype=float)
    return np.array([p[0], p[1], 1.0])


def affine(P):
    P = np.asarray(P, dtype=float)
    if abs(P[2]) < 1e-12:
        return None
    return P[:2] / P[2]


def circle_from_three(points: np.ndarray):
    pts = np.asarray(points, dtype=float)
    mat = np.column_stack([pts[:, 0], pts[:, 1], np.ones(len(pts))])
    rhs = -(pts[:, 0] ** 2 + pts[:, 1] ** 2)
    d, e, f = np.linalg.solve(mat, rhs)
    center = np.array([-d / 2, -e / 2])
    radius = math.sqrt(max(0.0, float(center @ center - f)))
    return center, radius


def fit_circle(points: np.ndarray):
    pts = np.asarray(points, dtype=float)
    mat = np.column_stack([pts[:, 0], pts[:, 1], np.ones(len(pts))])
    rhs = -(pts[:, 0] ** 2 + pts[:, 1] ** 2)
    d, e, f = np.linalg.lstsq(mat, rhs, rcond=None)[0]
    center = np.array([-d / 2, -e / 2])
    radius = math.sqrt(max(0.0, float(center @ center - f)))
    residual = np.sqrt(np.mean((np.sum((pts - center) ** 2, axis=1) - radius**2) ** 2))
    return center, radius, float(residual)


def split_on_jumps(points: np.ndarray, max_jump: float = 0.18):
    pts = np.asarray(points, dtype=float)
    if len(pts) == 0:
        return []
    jumps = np.where(np.linalg.norm(np.diff(pts, axis=0), axis=1) > max_jump)[0]
    starts = [0] + [int(j + 1) for j in jumps]
    ends = [int(j + 1) for j in jumps] + [len(pts)]
    return [pts[s:e] for s, e in zip(starts, ends) if e - s > 2]


def geodesic_arc_points(p, q, samples: int = 160) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    if np.linalg.norm(p - q) < 1e-12:
        return np.repeat(p[None, :], samples, axis=0)
    cross0 = p[0] * q[1] - p[1] * q[0]
    if abs(cross0) < 1e-10:
        return np.linspace(p, q, samples)
    mat = 2 * np.vstack([p, q])
    rhs = np.array([p @ p + 1, q @ q + 1], dtype=float)
    center = np.linalg.solve(mat, rhs)
    radius = math.sqrt(max(0.0, float(center @ center - 1)))
    t0 = math.atan2(p[1] - center[1], p[0] - center[0])
    t1 = math.atan2(q[1] - center[1], q[0] - center[0])
    delta = (t1 - t0) % (2 * np.pi)
    theta_a = np.linspace(t0, t0 + delta, samples)
    theta_b = np.linspace(t0, t0 - (2 * np.pi - delta), samples)
    arc_a = np.column_stack([center[0] + radius * np.cos(theta_a), center[1] + radius * np.sin(theta_a)])
    arc_b = np.column_stack([center[0] + radius * np.cos(theta_b), center[1] + radius * np.sin(theta_b)])
    score_a = np.nanmax(np.linalg.norm(arc_a, axis=1))
    score_b = np.nanmax(np.linalg.norm(arc_b, axis=1))
    if score_a <= 1 + 1e-7 and score_b > 1 + 1e-7:
        return arc_a
    if score_b <= 1 + 1e-7 and score_a > 1 + 1e-7:
        return arc_b
    return arc_a if np.mean(np.linalg.norm(arc_a, axis=1)) <= np.mean(np.linalg.norm(arc_b, axis=1)) else arc_b


def tangent_toward(p, q) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    cross0 = p[0] * q[1] - p[1] * q[0]
    if abs(cross0) < 1e-10:
        v = q - p
        return v / np.linalg.norm(v)
    center = np.linalg.solve(2 * np.vstack([p, q]), np.array([p @ p + 1, q @ q + 1], dtype=float))
    radius_vector = p - center
    tangent = np.array([-radius_vector[1], radius_vector[0]])
    if tangent @ (q - p) < 0:
        tangent = -tangent
    return tangent / np.linalg.norm(tangent)


def poincare_angle(vertex, p, q) -> float:
    u = tangent_toward(vertex, p)
    v = tangent_toward(vertex, q)
    dot = float(np.clip(u @ v, -1.0, 1.0))
    angle = math.acos(dot)
    return min(angle, 2 * math.pi - angle)


def triangle_angles(vertices: np.ndarray):
    a, b, c = np.asarray(vertices, dtype=float)
    return [poincare_angle(a, b, c), poincare_angle(b, c, a), poincare_angle(c, a, b)]


def plot_geodesic(ax, p, q, **kwargs):
    pts = geodesic_arc_points(p, q)
    ax.plot(pts[:, 0], pts[:, 1], **kwargs)
    return pts


def hyperbolic_circle_points(center: complex, radius: float, samples: int = 320) -> np.ndarray:
    rho = math.tanh(radius / 2)
    theta = np.linspace(0, 2 * np.pi, samples, endpoint=False)
    w = rho * np.exp(1j * theta)
    z = (w + center) / (np.conj(center) * w + 1)
    return np.column_stack([z.real, z.imag])


def disk_translate(z: complex, a: complex) -> complex:
    return (z + a) / (np.conj(a) * z + 1)


def rotate(z: complex, theta: float) -> complex:
    return complex(math.cos(theta), math.sin(theta)) * z


def regular_ngon_parameters(n: int, m: int):
    alpha = math.pi / n
    beta = math.pi / m
    exists = (1 / n + 1 / m) < 0.5
    if not exists:
        return {"n": n, "m": m, "exists_hyperbolic": False, "alpha": alpha, "beta": beta}
    sinh_c_sq = 1 / (math.tan(alpha) ** 2 * math.tan(beta) ** 2) - 1
    sinh_c = math.sqrt(max(0.0, sinh_c_sq))
    circumradius = math.asinh(sinh_c)
    half_side = math.asinh(math.sin(alpha) * sinh_c)
    inradius = math.asinh(math.sin(beta) * sinh_c)
    return {
        "n": n,
        "m": m,
        "exists_hyperbolic": True,
        "vertex_angle_deg": 360 / m,
        "alpha_deg": math.degrees(alpha),
        "beta_deg": math.degrees(beta),
        "circumradius": circumradius,
        "inradius": inradius,
        "side_length": 2 * half_side,
        "euclidean_vertex_radius": math.tanh(circumradius / 2),
    }


def regular_ngon_vertices(n: int, m: int) -> np.ndarray:
    params = regular_ngon_parameters(n, m)
    if not params["exists_hyperbolic"]:
        raise ValueError("The pair does not satisfy the hyperbolic tiling inequality")
    rho = float(params["euclidean_vertex_radius"])
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False) + np.pi / 2
    return np.column_stack([rho * np.cos(theta), rho * np.sin(theta)])


def pairwise_hdist(points):
    vals = []
    arr = [np.array([z.real, z.imag]) for z in points]
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            vals.append(poincare_distance(arr[i], arr[j]))
    return np.array(vals, dtype=float)


def record_image_stats(paths):
    stats = []
    for path in paths:
        item = image_stats(path)
        item["path"] = rel(path)
        stats.append(item)
    return stats


print("Helpers ready for", CHAPTER_SLUG)

## 26.1 Circles and Cycles in the Poincare Disk

The Poincare disk makes some hyperbolic cycles look deceptively Euclidean. Proper hyperbolic circles and horocycles are drawn as Euclidean circles, but their metric centers are not usually Euclidean centers. Hypercycles expose the algebraic side of the story: a Euclidean circle that crosses the boundary contributes one visible branch and, after inversion in the absolute, a companion branch.

The first panel separates those three cases. The second panel turns the chapter's A-circle/C-circle distinction into an experiment: through three disk points there is one C-circle, but four algebraic A-circles once each point may be replaced by its inversion outside the disk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.0))
ax = axes[0]
disk_boundary(ax)
ax.set_title("Cycle types in the Poincare disk")

proper = hyperbolic_circle_points(center=0.34 - 0.16j, radius=0.95)
euc_center, euc_radius, circle_fit_residual = fit_circle(proper)
ax.plot(proper[:, 0], proper[:, 1], color="#2364aa", lw=2.2, label="proper real circle")
ax.scatter([0.34], [-0.16], color="#2364aa", s=36, marker="x", label="hyperbolic center")
ax.scatter([euc_center[0]], [euc_center[1]], color="#6baed6", s=25, label="Euclidean center")

theta = np.linspace(0, 2 * np.pi, 360)
horo_center = np.array([0.72, 0.0])
horo_radius = 0.28
horo = np.column_stack([horo_center[0] + horo_radius * np.cos(theta), horo_center[1] + horo_radius * np.sin(theta)])
ax.plot(horo[:, 0], horo[:, 1], color="#c43b3b", lw=2.0, label="horocycle")
ax.scatter([1.0], [0.0], color="#c43b3b", s=30, marker="o", label="ideal center")

hyper_center = np.array([-0.18, -0.76])
hyper_radius = 0.64
full = np.column_stack([hyper_center[0] + hyper_radius * np.cos(theta), hyper_center[1] + hyper_radius * np.sin(theta)])
radii = np.linalg.norm(full, axis=1)
inside_segments = split_on_jumps(full[radii < 1])
outside = full[radii > 1]
inverted = outside / np.sum(outside**2, axis=1)[:, None]
inverted_segments = split_on_jumps(inverted)
for j, segment in enumerate(inside_segments):
    ax.plot(segment[:, 0], segment[:, 1], color="#2a9d55", lw=2.1, label="hypercycle branch" if j == 0 else None)
for j, segment in enumerate(inverted_segments):
    ax.plot(segment[:, 0], segment[:, 1], color="#7bc96f", lw=2.1, ls="--", label="inverted companion" if j == 0 else None)
ax.legend(loc="lower left", fontsize=7, frameon=True)

ax = axes[1]
disk_boundary(ax)
ax.set_title("Three points: one C-circle, four A-circles")
points = np.array([[-0.52, -0.20], [0.12, 0.42], [0.48, -0.10]])
for label, p in zip(["A", "B", "C"], points):
    ax.scatter([p[0]], [p[1]], color="black", s=28, zorder=5)
    ax.text(p[0] + 0.03, p[1] + 0.03, label, fontsize=9, weight="bold")

c_center, c_radius = circle_from_three(points)
c_full = np.column_stack([c_center[0] + c_radius * np.cos(theta), c_center[1] + c_radius * np.sin(theta)])
for j, segment in enumerate(split_on_jumps(c_full[np.linalg.norm(c_full, axis=1) < 1], max_jump=0.12)):
    ax.plot(segment[:, 0], segment[:, 1], color="black", lw=2.2, ls=":", label="unique C-circle" if j == 0 else None)

colors = ["#4c78a8", "#f58518", "#54a24b", "#b279a2"]
choices = [(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1)]
a_circle_summaries = []
for idx, bits in enumerate(choices):
    selected = []
    for bit, point in zip(bits, points):
        selected.append(point if bit == 0 else point / float(point @ point))
    center, radius = circle_from_three(np.asarray(selected))
    sample = np.column_stack([center[0] + radius * np.cos(theta), center[1] + radius * np.sin(theta)])
    sample_r = np.linalg.norm(sample, axis=1)
    mapped = sample.copy()
    outside_mask = sample_r > 1
    mapped[outside_mask] = sample[outside_mask] / np.sum(sample[outside_mask] ** 2, axis=1)[:, None]
    mapped = mapped[np.linalg.norm(mapped, axis=1) < 1.0001]
    for j, segment in enumerate(split_on_jumps(mapped, max_jump=0.16)):
        ax.plot(segment[:, 0], segment[:, 1], color=colors[idx], lw=1.6, alpha=0.95, label=f"A-circle {idx + 1}" if j == 0 else None)
    a_circle_summaries.append({"choice": "".join(map(str, bits)), "euclidean_center_x": center[0], "euclidean_center_y": center[1], "euclidean_radius": radius})
ax.legend(loc="lower right", fontsize=7, frameon=True)

cycle_path = save_fig(fig, "cycle-classification-and-a-circles.png")
cycle_table_path = save_csv(pd.DataFrame([
    {"cycle": "proper real circle", "poincare_picture": "Euclidean circle inside the disk", "center_location": "inside", "branches_visible": 1, "inspection_target": "compare the metric center with the Euclidean center to see the Poincare distortion"},
    {"cycle": "horocycle", "poincare_picture": "Euclidean circle tangent to the boundary", "center_location": "on boundary", "branches_visible": 1, "inspection_target": "check that the ideal center is the tangency point on the absolute"},
    {"cycle": "hypercycle", "poincare_picture": "two inversion-related circular arcs", "center_location": "outside", "branches_visible": 2, "inspection_target": "trace the outside arc through inversion to obtain the companion branch"},
]), "cycle-classification.csv")
a_circle_table_path = save_csv(pd.DataFrame(a_circle_summaries), "a-circle-inversion-choices.csv")

check_data["cycles"] = {
    "proper_circle_fit_residual": circle_fit_residual,
    "a_circle_count": len(a_circle_summaries),
    "c_circle_radius": c_radius,
    "a_circle_table": rel(a_circle_table_path),
}

display_artifact(cycle_path, width=900)

## 26.2 Area and Angle Defect

In the disk model, geodesic sides are circular arcs orthogonal to the boundary. The model is conformal, so the angle at a finite vertex is the ordinary Euclidean angle between tangent directions. That makes the defect

`area = pi - alpha - beta - gamma`

computable directly from the diagram. The normalization used here is the chapter's normalization: an ideal triangle has area `pi`.

In [ ]:
triangles = {
    "small near origin": np.array([[0.00, 0.00], [0.22, 0.00], [0.00, 0.22]]),
    "moderate": np.array([[-0.42, -0.12], [0.52, -0.05], [0.02, 0.62]]),
    "near ideal": np.array([[-0.82, -0.18], [0.78, -0.12], [0.08, 0.88]]),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
area_rows = []
for ax, (name, verts) in zip(axes, triangles.items()):
    disk_boundary(ax)
    boundary_path = []
    for i, j in [(0, 1), (1, 2), (2, 0)]:
        arc = plot_geodesic(ax, verts[i], verts[j], color="#305f72", lw=2)
        boundary_path.extend(arc.tolist() if not boundary_path else arc[1:].tolist())
    boundary_path = np.asarray(boundary_path)
    ax.fill(boundary_path[:, 0], boundary_path[:, 1], color="#9ecae1", alpha=0.35)
    ax.scatter(verts[:, 0], verts[:, 1], color="#08306b", s=24, zorder=5)
    angles = triangle_angles(verts)
    defect = math.pi - sum(angles)
    area_rows.append({
        "triangle": name,
        "angle_a_deg": math.degrees(angles[0]),
        "angle_b_deg": math.degrees(angles[1]),
        "angle_c_deg": math.degrees(angles[2]),
        "angle_sum_deg": math.degrees(sum(angles)),
        "area_defect": defect,
    })
    ax.set_title(f"{name}\ndefect = {defect:.3f}")

area_path = save_fig(fig, "angle-defect-area-triangles.png")
area_table = pd.DataFrame(area_rows)
area_table_path = save_csv(area_table, "angle-defect-triangle-ledger.csv")

a = sp.symbols("a", positive=True)
defect_expr = sp.pi / 2 - 2 * sp.acos(1 / sp.sqrt(2 - a**2))
small_series = sp.series(defect_expr, a, 0, 6).removeO()
small_rows = []
for aval in [0.04, 0.08, 0.16, 0.24, 0.32]:
    defect_val = float(defect_expr.subs(a, aval).evalf())
    euclidean_val = aval**2 / 2
    leg_length = -0.5 * math.log((1 - aval) / (1 + aval))
    small_rows.append({
        "a_coordinate": aval,
        "hyperbolic_leg_length": leg_length,
        "defect_formula": defect_val,
        "euclidean_a_squared_over_2": euclidean_val,
        "relative_error_to_euclidean": abs(defect_val - euclidean_val) / euclidean_val,
    })
small_table_path = save_csv(pd.DataFrame(small_rows), "small-triangle-defect-asymptotics.csv")

check_data["area"] = {
    "all_defects_positive": bool((area_table["area_defect"] > 0).all()),
    "near_ideal_largest_defect": area_table.sort_values("area_defect").iloc[-1]["triangle"],
    "small_defect_series": str(small_series),
    "smallest_relative_error": min(row["relative_error_to_euclidean"] for row in small_rows),
}

display_artifact(area_path, width=900)
area_table

## 26.3 Thales and Pythagoras

The hyperbolic peripheral-angle theorem does not survive unchanged: a generic fixed-angle locus is more complicated than a circle. The right-angle case is special. In the Beltrami-Klein disk, choose a line through one endpoint, take its pole with respect to the absolute, and join that pole to the other endpoint. The intersection point sweeps a conic.

For right triangles, the distance identity is not `a^2 + b^2 = c^2`; it is `cosh(a) cosh(b) = cosh(c)`. Rewriting with `sinh` shows the correction term that disappears only in the small-length limit.

In [ ]:
A = np.array([-0.46, -0.14])
B = np.array([0.48, 0.18])
Ah = hpoint_xy(A)
Bh = hpoint_xy(B)
raw_points = []
orthogonality_residuals = []
sample_pairs = []

for angle in np.linspace(0.02, np.pi - 0.02, 260):
    direction = np.array([math.cos(angle), math.sin(angle)])
    l = np.cross(Ah, hpoint_xy(A + direction))
    pole = ABSOLUTE_CONIC @ l
    g = np.cross(Bh, pole)
    X = np.cross(l, g)
    x = affine(X)
    if x is not None and np.linalg.norm(x) < 0.995:
        raw_points.append(x)
        orthogonality_residuals.append(float(l @ ABSOLUTE_CONIC @ g))
        if len(sample_pairs) < 6 and len(raw_points) % 28 == 0:
            sample_pairs.append((l, g, x))
raw_points = np.asarray(raw_points)

M = np.column_stack([
    raw_points[:, 0] ** 2,
    raw_points[:, 0] * raw_points[:, 1],
    raw_points[:, 1] ** 2,
    raw_points[:, 0],
    raw_points[:, 1],
    np.ones(len(raw_points)),
])
_, _, vh = np.linalg.svd(M)
conic_coeff = vh[-1, :]
conic_rms = float(np.sqrt(np.mean((M @ conic_coeff) ** 2)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
disk_boundary(ax)
ax.set_title("Hyperbolic Thales locus in Klein coordinates")
ax.plot([A[0], B[0]], [A[1], B[1]], color="black", lw=2.2, label="given segment")
ax.scatter([A[0], B[0]], [A[1], B[1]], color="black", s=30)
ax.text(A[0] - 0.06, A[1] - 0.05, "A", weight="bold")
ax.text(B[0] + 0.03, B[1] + 0.02, "B", weight="bold")
ax.scatter(raw_points[:, 0], raw_points[:, 1], s=8, color="#cc503e", alpha=0.55, label="right-angle intersections")
xx, yy = np.meshgrid(np.linspace(-1.03, 1.03, 260), np.linspace(-1.03, 1.03, 260))
F = conic_coeff[0] * xx**2 + conic_coeff[1] * xx * yy + conic_coeff[2] * yy**2 + conic_coeff[3] * xx + conic_coeff[4] * yy + conic_coeff[5]
F = np.where(xx**2 + yy**2 <= 1.02, F, np.nan)
ax.contour(xx, yy, F, levels=[0], colors=["#7a1f1f"], linewidths=2.0)
for l, g, x in sample_pairs:
    for line, color in [(l, "#777777"), (g, "#aaaaaa")]:
        if abs(line[1]) > 1e-9:
            xs = np.array([-1.05, 1.05])
            ys = -(line[0] * xs + line[2]) / line[1]
        else:
            xs = np.full(2, -line[2] / line[0])
            ys = np.array([-1.05, 1.05])
        ax.plot(xs, ys, color=color, lw=0.7, alpha=0.45)
ax.legend(loc="lower left", fontsize=7, frameon=True)

right_rows = []
for px, qy in [(0.18, 0.24), (0.32, 0.38), (0.48, 0.42), (0.62, 0.52)]:
    p = np.array([px, 0.0])
    q = np.array([0.0, qy])
    r = np.array([0.0, 0.0])
    da = poincare_distance(r, p)
    db = poincare_distance(r, q)
    dc = poincare_distance(p, q)
    sinh_sum = math.sinh(da) ** 2 + math.sinh(db) ** 2
    correction = math.sinh(da) ** 2 * math.sinh(db) ** 2
    right_rows.append({
        "leg_a": da,
        "leg_b": db,
        "hypotenuse_c": dc,
        "cosh_product_residual": abs(math.cosh(da) * math.cosh(db) - math.cosh(dc)),
        "sinh2_a_plus_sinh2_b": sinh_sum,
        "correction_term": correction,
        "sinh2_c": math.sinh(dc) ** 2,
        "corrected_residual": abs(sinh_sum + correction - math.sinh(dc) ** 2),
    })
right_df = pd.DataFrame(right_rows)

ax = axes[1]
ax.set_title("Pythagorean correction term")
idx = np.arange(len(right_df))
ax.bar(idx - 0.17, right_df["sinh2_a_plus_sinh2_b"], width=0.34, color="#4c78a8", label="sinh^2(a)+sinh^2(b)")
ax.bar(idx + 0.17, right_df["correction_term"], bottom=right_df["sinh2_a_plus_sinh2_b"], width=0.34, color="#f58518", label="plus product correction")
ax.plot(idx + 0.17, right_df["sinh2_c"], color="black", marker="o", lw=1.5, label="sinh^2(c)")
ax.set_xticks(idx)
ax.set_xticklabels([f"case {i+1}" for i in idx])
ax.set_ylabel("value")
ax.legend(fontsize=7)

thales_pythagoras_path = save_fig(fig, "thales-conic-and-pythagoras-correction.png")
pythagoras_table_path = save_csv(right_df, "pythagoras-right-triangle-checks.csv")

x, y = sp.symbols("x y")
Asp = sp.diag(1, 1, -1)
ps = sp.Matrix([x, 0, 1])
qs = sp.Matrix([0, y, 1])
rs = sp.Matrix([0, 0, 1])
omega = lambda u, v: (u.T * Asp * v)[0]
projective_identity = sp.simplify(omega(ps, rs) * omega(rs, qs) - omega(ps, qs) * omega(rs, rs))

check_data["thales_pythagoras"] = {
    "thales_points_sampled": int(len(raw_points)),
    "thales_conic_rms_residual": conic_rms,
    "orthogonality_max_residual": float(np.max(np.abs(orthogonality_residuals))),
    "pythagoras_max_cosh_residual": float(right_df["cosh_product_residual"].max()),
    "pythagoras_max_corrected_residual": float(right_df["corrected_residual"].max()),
    "origin_right_triangle_projective_identity": str(projective_identity),
}

display_artifact(thales_pythagoras_path, width=900)
right_df

## 26.4 Constructing Regular n-Gons

A regular hyperbolic n-gon with vertex angle `psi = 2*pi/m` is built from `2n` congruent right triangles. The triangle has angles `alpha = pi/n`, `beta = pi/m`, and a right angle at the side midpoint. The chapter's computation gives the circumradius `c` through

`sinh(c)^2 = 1/(tan(alpha)^2 tan(beta)^2) - 1`.

The expression is real exactly in the hyperbolic tiling range `1/n + 1/m < 1/2`. Equality is the Euclidean boundary case.

In [ ]:
ngon_pairs = [(5, 4), (6, 4), (7, 3), (8, 3), (9, 4), (12, 3)]
ngon_df = pd.DataFrame([regular_ngon_parameters(n, m) for n, m in ngon_pairs])
ngon_table_path = save_csv(ngon_df, "regular-ngon-measurements.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
disk_boundary(ax)
ax.set_title("Right-angled pentagon from 10 right triangles")
pentagon_verts = regular_ngon_vertices(5, 4)
for i in range(len(pentagon_verts)):
    plot_geodesic(ax, pentagon_verts[i], pentagon_verts[(i + 1) % len(pentagon_verts)], color="#2f5d50", lw=2.0)
ax.scatter(pentagon_verts[:, 0], pentagon_verts[:, 1], color="#2f5d50", s=25, zorder=5)
for v in pentagon_verts:
    ax.plot([0, v[0]], [0, v[1]], color="#8ab17d", lw=0.7, alpha=0.55)
edge01 = geodesic_arc_points(pentagon_verts[0], pentagon_verts[1], samples=161)
mid = edge01[len(edge01) // 2]
ax.plot([0, pentagon_verts[0, 0]], [0, pentagon_verts[0, 1]], color="#e76f51", lw=2.0)
ax.plot([0, mid[0]], [0, mid[1]], color="#e76f51", lw=1.6, ls="--")
ax.plot([mid[0], pentagon_verts[0, 0]], [mid[1], pentagon_verts[0, 1]], color="#e76f51", lw=1.6, ls="--")
ax.scatter([0, mid[0]], [0, mid[1]], color="#e76f51", s=22)
ax.text(0.04, 0.04, "alpha=pi/5", color="#7f1d1d", fontsize=8)
ax.text(pentagon_verts[0, 0] - 0.18, pentagon_verts[0, 1] - 0.04, "beta=pi/4", color="#7f1d1d", fontsize=8)

ax = axes[1]
ns = range(3, 13)
ms = range(3, 9)
colors = {-1: "#7fc97f", 0: "#fddc7a", 1: "#f29e9e"}
for i, m in enumerate(ms):
    for j, n in enumerate(ns):
        value = 1 / n + 1 / m
        cls = -1 if value < 0.5 else (0 if abs(value - 0.5) < 1e-12 else 1)
        ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=colors[cls], edgecolor="white"))
        ax.text(j + 0.5, i + 0.5, f"{n},{m}", ha="center", va="center", fontsize=7)
ax.set_xlim(0, len(ns))
ax.set_ylim(0, len(ms))
ax.set_xticks(np.arange(len(ns)) + 0.5)
ax.set_xticklabels(list(ns))
ax.set_yticks(np.arange(len(ms)) + 0.5)
ax.set_yticklabels(list(ms))
ax.set_xlabel("n sides")
ax.set_ylabel("m polygons around a vertex")
ax.set_title("Tiling inequality: 1/n + 1/m < 1/2")
ax.invert_yaxis()
legend_handles = [plt.Rectangle((0, 0), 1, 1, facecolor=colors[k], edgecolor="white", label=v) for k, v in [(-1, "hyperbolic"), (0, "Euclidean"), (1, "spherical")]]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, frameon=True)

ngon_path = save_fig(fig, "regular-ngon-construction-and-tiling-range.png")

fig_html = go.Figure()
boundary_theta = np.linspace(0, 2 * np.pi, 360)
fig_html.add_trace(go.Scatter(x=np.cos(boundary_theta), y=np.sin(boundary_theta), mode="lines", name="absolute", line=dict(color="black", width=2), hoverinfo="skip"))
trace_groups = []
for pair_index, (n, m) in enumerate(ngon_pairs):
    start_trace = len(fig_html.data)
    verts = regular_ngon_vertices(n, m)
    xs, ys = [], []
    for i in range(n):
        arc = geodesic_arc_points(verts[i], verts[(i + 1) % n], samples=90)
        xs.extend(arc[:, 0].tolist() + [None])
        ys.extend(arc[:, 1].tolist() + [None])
    params = regular_ngon_parameters(n, m)
    fig_html.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=f"{{{n},{m}}}", visible=(pair_index == 0), line=dict(width=3), hovertemplate=f"n={n}<br>m={m}<br>side={params['side_length']:.3f}<br>radius={params['circumradius']:.3f}<extra></extra>"))
    fig_html.add_trace(go.Scatter(x=verts[:, 0], y=verts[:, 1], mode="markers", name=f"vertices {n},{m}", visible=(pair_index == 0), marker=dict(size=7), hoverinfo="skip"))
    trace_groups.append((start_trace, len(fig_html.data)))
buttons = []
for pair_index, (n, m) in enumerate(ngon_pairs):
    visible = [True] + [False] * (len(fig_html.data) - 1)
    start, end = trace_groups[pair_index]
    for k in range(start, end):
        visible[k] = True
    params = regular_ngon_parameters(n, m)
    buttons.append(dict(label=f"{{{n},{m}}}", method="update", args=[{"visible": visible}, {"title": f"Regular hyperbolic {{n,m}} = {{{n},{m}}}: side {params['side_length']:.3f}, radius {params['circumradius']:.3f}"}]))
fig_html.update_layout(title="Regular hyperbolic n-gon parameter lab", width=760, height=700, xaxis=dict(scaleanchor="y", range=[-1.05, 1.05], showgrid=False, zeroline=False), yaxis=dict(range=[-1.05, 1.05], showgrid=False, zeroline=False), updatemenus=[dict(buttons=buttons, x=0.02, y=1.08, xanchor="left", yanchor="top")], margin=dict(l=20, r=20, t=80, b=20))
ngon_html_path = save_plotly(fig_html, "regular-ngon-parameter-lab.html")

pentagon_angles = [poincare_angle(pentagon_verts[i], pentagon_verts[(i - 1) % 5], pentagon_verts[(i + 1) % 5]) for i in range(5)]
angle_error = max(abs(angle - math.pi / 2) for angle in pentagon_angles)
check_data["regular_ngons"] = {
    "measured_right_pentagon_angle_error": angle_error,
    "hyperbolic_pairs_listed": int(ngon_df["exists_hyperbolic"].sum()),
    "min_tiling_gap": float(min(0.5 - (1 / n + 1 / m) for n, m in ngon_pairs)),
}

display_artifact(ngon_path, width=900)
display_artifact(ngon_html_path, width="100%", height=620)
ngon_df

## 26.5 Symmetry Group Orbits

Hyperbolic symmetry drawings are not just decorative tessellations. A group element is an isometry, so it preserves hyperbolic distances even while Euclidean distances near the boundary appear compressed. The orbit experiment below uses two disk-preserving Mobius translations and their inverses. We draw only a finite word-depth sample, then check distance preservation and separation for that finite sample.

In [ ]:
translation_length = 1.18
t = math.tanh(translation_length / 2)


def Tx(z):
    return disk_translate(z, t)


def Txi(z):
    return disk_translate(z, -t)


def Ty(z):
    return rotate(disk_translate(rotate(z, -math.pi / 2), t), math.pi / 2)


def Tyi(z):
    return rotate(disk_translate(rotate(z, -math.pi / 2), -t), math.pi / 2)


generators = {"Tx": Tx, "Tx^-1": Txi, "Ty": Ty, "Ty^-1": Tyi}
max_depth = 5
seen = {(0, 0): (0 + 0j, 0, "e")}
frontier = deque([(0 + 0j, 0, "e")])
while frontier:
    z, depth, word = frontier.popleft()
    if depth >= max_depth:
        continue
    for name, fn in generators.items():
        w = fn(z)
        if abs(w) >= 0.999999:
            continue
        key = (round(w.real, 8), round(w.imag, 8))
        if key not in seen:
            seen[key] = (w, depth + 1, name if word == "e" else word + " " + name)
            frontier.append((w, depth + 1, seen[key][2]))

orbit = list(seen.values())
orbit_points = [item[0] for item in orbit]
orbit_depths = np.array([item[1] for item in orbit])
min_pair_distance = float(pairwise_hdist(orbit_points).min())

p0 = 0.11 + 0.19j
q0 = -0.23 + 0.16j
d0 = poincare_distance(np.array([p0.real, p0.imag]), np.array([q0.real, q0.imag]))
distance_preservation = {}
for name, fn in generators.items():
    p1 = fn(p0)
    q1 = fn(q0)
    d1 = poincare_distance(np.array([p1.real, p1.imag]), np.array([q1.real, q1.imag]))
    distance_preservation[name] = abs(d1 - d0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
disk_boundary(ax)
ax.set_title("Finite orbit under two hyperbolic translations")
sc = ax.scatter([z.real for z in orbit_points], [z.imag for z in orbit_points], c=orbit_depths, cmap="viridis", s=18, zorder=4)
ax.scatter([0], [0], color="white", edgecolor="black", s=45, zorder=6, label="seed")
ax.legend(loc="lower left", fontsize=7)
cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("word depth")

ax = axes[1]
G = nx.DiGraph()
G.add_edges_from([
    ("cycle classification", "A/C circle distinction"),
    ("inversion in absolute", "A/C circle distinction"),
    ("additivity", "triangle area defect"),
    ("motion invariance", "triangle area defect"),
    ("polarity", "hyperbolic Thales conic"),
    ("line orthogonality", "hyperbolic Thales conic"),
    ("bilinear absolute form", "Pythagoras identity"),
    ("right-triangle decomposition", "regular n-gon"),
    ("regular n-gon", "tiling inequality"),
    ("disk Mobius isometries", "symmetry orbits"),
])
pos = nx.spring_layout(G, seed=7, k=0.9)
nx.draw_networkx_edges(G, pos, ax=ax, arrowstyle="-|>", arrowsize=10, width=1.0, edge_color="#777777")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f6f1d1", edgecolors="#555555", node_size=1200)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
ax.set_title("Proof and invariant dependencies")
ax.axis("off")

orbit_path = save_fig(fig, "symmetry-orbit-and-proof-dependencies.png")
orbit_df = pd.DataFrame({"real": [z.real for z in orbit_points], "imag": [z.imag for z in orbit_points], "word_depth": orbit_depths, "radius": [abs(z) for z in orbit_points]})
orbit_table_path = save_csv(orbit_df, "symmetry-orbit-sample.csv")

fig_orbit = go.Figure()
fig_orbit.add_trace(go.Scatter(x=np.cos(boundary_theta), y=np.sin(boundary_theta), mode="lines", line=dict(color="black"), name="absolute", hoverinfo="skip"))
fig_orbit.add_trace(go.Scatter(x=[z.real for z in orbit_points], y=[z.imag for z in orbit_points], mode="markers", marker=dict(size=7, color=orbit_depths, colorscale="Viridis", colorbar=dict(title="depth")), text=[item[2] for item in orbit], hovertemplate="%{text}<br>r=%{customdata:.3f}<extra></extra>", customdata=np.array([abs(z) for z in orbit_points]), name="orbit sample"))
fig_orbit.update_layout(title="Mobius-isometry orbit sample", width=760, height=700, xaxis=dict(scaleanchor="y", range=[-1.05, 1.05], showgrid=False, zeroline=False), yaxis=dict(range=[-1.05, 1.05], showgrid=False, zeroline=False), margin=dict(l=20, r=20, t=60, b=20))
orbit_html_path = save_plotly(fig_orbit, "symmetry-orbit-depth-lab.html")

check_data["symmetry"] = {
    "orbit_point_count": len(orbit_points),
    "min_pairwise_hyperbolic_distance_depth_5": min_pair_distance,
    "max_generator_distance_residual": float(max(distance_preservation.values())),
    "distance_preservation_residuals": distance_preservation,
}

display_artifact(orbit_path, width=900)
display_artifact(orbit_html_path, width="100%", height=620)

## Artifact Gallery

The generated files are displayed inline near the section that explains them. These direct links keep the notebook readable when opened before execution.

![Cycle classification and A-circles](../../artifacts/chapter-26-selected-topics-in-hyperbolic-geometry/figures/cycle-classification-and-a-circles.png)

![Angle defect triangles](../../artifacts/chapter-26-selected-topics-in-hyperbolic-geometry/figures/angle-defect-area-triangles.png)

![Thales and Pythagoras](../../artifacts/chapter-26-selected-topics-in-hyperbolic-geometry/figures/thales-conic-and-pythagoras-correction.png)

![Regular n-gon construction](../../artifacts/chapter-26-selected-topics-in-hyperbolic-geometry/figures/regular-ngon-construction-and-tiling-range.png)

![Symmetry orbit and dependencies](../../artifacts/chapter-26-selected-topics-in-hyperbolic-geometry/figures/symmetry-orbit-and-proof-dependencies.png)

## Applied Lab: Design a Hyperbolic Polygon Cell

Use the table below as a compact design lab. Pick integers `n` and `m`; the intended cell has `n` sides and vertex angle `2*pi/m`. The lab reports whether the pair is spherical, Euclidean, or hyperbolic, and when hyperbolic it computes the side length and circumradius. Compare rows close to the Euclidean boundary with rows far from it: near the boundary the radius is small, while deeper hyperbolic choices push the vertices toward the absolute.

In [ ]:
def polygon_cell_report(pairs):
    rows = []
    for n, m in pairs:
        value = 1 / n + 1 / m
        if value < 0.5:
            params = regular_ngon_parameters(n, m)
            rows.append({
                "n": n,
                "m": m,
                "geometry": "hyperbolic",
                "tiling_gap": 0.5 - value,
                "vertex_angle_deg": params["vertex_angle_deg"],
                "side_length": params["side_length"],
                "circumradius": params["circumradius"],
                "euclidean_vertex_radius": params["euclidean_vertex_radius"],
            })
        elif abs(value - 0.5) < 1e-12:
            rows.append({"n": n, "m": m, "geometry": "Euclidean boundary", "tiling_gap": 0.0})
        else:
            rows.append({"n": n, "m": m, "geometry": "spherical side", "tiling_gap": 0.5 - value})
    return pd.DataFrame(rows)


lab_pairs = [(3, 6), (4, 4), (5, 4), (6, 3), (7, 3), (8, 3), (12, 5)]
lab_df = polygon_cell_report(lab_pairs)
lab_table_path = save_csv(lab_df, "polygon-cell-design-lab.csv")
check_data["lab"] = {
    "lab_rows": len(lab_df),
    "hyperbolic_lab_rows": int((lab_df["geometry"] == "hyperbolic").sum()),
    "max_reported_radius": float(pd.to_numeric(lab_df.get("euclidean_vertex_radius"), errors="coerce").max()),
}
lab_df

## Takeaways

- In the Poincare disk, a curve can look like an ordinary circle while its hyperbolic center, branch count, or algebraic completion tells a different story.
- Area is controlled by angle defect. The formula is exact for all geodesic triangles and agrees with Euclidean area only as a local limit.
- Hyperbolic Thales survives in the right-angle case because polarity turns a pencil of lines into another pencil, whose intersections form a conic.
- Hyperbolic Pythagoras is multiplicative in `cosh`; the additive Euclidean form reappears only after expanding the `sinh` correction for small lengths.
- Regular n-gons and symmetry orbits are plentiful because hyperbolic angle sums have room: inequalities replace Euclidean equalities.

In [ ]:
all_artifacts = list(artifact_registry.values())
assert len(all_artifacts) >= 10
assert_artifacts(all_artifacts, min_size=256)

png_paths = [path for path in all_artifacts if path.suffix.lower() == ".png"]
html_paths = [path for path in all_artifacts if path.suffix.lower() == ".html"]
table_paths = [path for path in all_artifacts if path.suffix.lower() == ".csv"]

sample = [-1.4, -0.2, 0.75, 1.6]
image = [(1.1 * x - 0.25) / (0.22 * x + 1.0) for x in sample]
cross_ratio_error = abs(cross_ratio(*sample) - cross_ratio(*image))

assert check_data["cycles"]["a_circle_count"] == 4
assert check_data["cycles"]["proper_circle_fit_residual"] < 1e-12
assert check_data["area"]["all_defects_positive"]
assert "a**2/2" in check_data["area"]["small_defect_series"]
assert check_data["thales_pythagoras"]["thales_conic_rms_residual"] < 1e-10
assert check_data["thales_pythagoras"]["orthogonality_max_residual"] < 1e-10
assert check_data["thales_pythagoras"]["pythagoras_max_cosh_residual"] < 1e-10
assert check_data["thales_pythagoras"]["pythagoras_max_corrected_residual"] < 1e-10
assert check_data["thales_pythagoras"]["origin_right_triangle_projective_identity"] == "0"
assert check_data["regular_ngons"]["measured_right_pentagon_angle_error"] < 1e-5
assert check_data["regular_ngons"]["min_tiling_gap"] > 0
assert check_data["symmetry"]["orbit_point_count"] > 20
assert check_data["symmetry"]["max_generator_distance_residual"] < 1e-10
assert cross_ratio_error < 1e-9

storyboard = {
    "chapter goal": "Read Euclidean-looking disk pictures with hyperbolic measurement: cycles, area defect, right-triangle identities, regular polygons, and symmetry orbits.",
    "source span read": "Perspectives on Projective Geometry, Chapter 26, sections 26.1-26.5; printed pp. 505-524; PDF pp. 527-546.",
    "concept inventory": [
        "proper real circles, horocycles, hypercycles, and the A-circle/C-circle distinction",
        "hyperbolic area as triangle angle defect with ideal-triangle normalization",
        "right-angle peripheral locus as a conic via polarity",
        "hyperbolic Pythagoras in cosh form and the sinh correction term",
        "regular n-gons from right-triangle decomposition and the tiling inequality",
        "finite samples of group orbits under disk-preserving Mobius isometries",
    ],
    "library routing table": [
        {"concept": "disk cycles and geodesic triangles", "representation": "static Poincare/Klein diagrams", "library": "matplotlib + numpy", "why": "precise 2D constructions with labels and stable PNG artifacts"},
        {"concept": "regular polygon and orbit parameter experiments", "representation": "standalone interactive disk views", "library": "plotly", "why": "hover/dropdown inspection of parameter-dependent hyperbolic measurements"},
        {"concept": "small-triangle and projective identities", "representation": "exact algebra", "library": "sympy", "why": "series and bilinear identities should be checked symbolically"},
        {"concept": "proof dependencies", "representation": "directed dependency graph", "library": "networkx", "why": "proof inputs and theorem outputs are graph-like rather than metric data"},
        {"concept": "numeric ledgers", "representation": "CSV tables", "library": "pandas", "why": "keeps validation values inspectable outside the notebook"},
    ],
    "visual sequence": [
        {"artifact_filename": rel(artifact_registry["cycle-classification-and-a-circles.png"]), "concept": "cycle classification and four A-circles", "inspection_target": "compare center location, branch count, and inversion choices", "validation": "circle fit residual and A-circle count"},
        {"artifact_filename": rel(artifact_registry["angle-defect-area-triangles.png"]), "concept": "area equals angle defect", "inspection_target": "defect increases as vertices move toward the absolute", "validation": "all sampled defects are positive; small-triangle series begins with a^2/2"},
        {"artifact_filename": rel(artifact_registry["thales-conic-and-pythagoras-correction.png"]), "concept": "Thales conic and Pythagoras correction", "inspection_target": "right-angle intersections fit a conic; sinh correction closes the distance identity", "validation": "orthogonality, conic, cosh, and corrected-sinh residuals"},
        {"artifact_filename": rel(artifact_registry["regular-ngon-construction-and-tiling-range.png"]), "concept": "regular n-gon construction", "inspection_target": "right-triangle wedge and hyperbolic tiling inequality", "validation": "measured pentagon angle and positive tiling gaps"},
        {"artifact_filename": rel(artifact_registry["symmetry-orbit-and-proof-dependencies.png"]), "concept": "symmetry group orbit and proof dependencies", "inspection_target": "orbit depth pushes points toward the boundary while isometries preserve distances", "validation": "distance preservation and finite-orbit separation"},
    ],
    "artifact plan": [rel(path) for path in all_artifacts],
    "computational checks": check_data,
    "proof-visualization strategy": "Use a dependency graph plus symbolic/numeric invariants: additivity and motion invariance feed area defect; polarity feeds Thales; the absolute bilinear form feeds Pythagoras.",
    "risks": "The orbit lab is a finite sample of a Mobius-generated group, not a proof of a full tessellation classification.",
    "acceptance criteria": "Notebook executes with nbclient; generated artifacts are nonblank and book-local; final checks assert the chapter identities and artifact integrity.",
}

visual_checks = {
    "chapter": 26,
    "all_files_exist": all(path.exists() for path in all_artifacts),
    "visual_count": len(all_artifacts),
    "raster_artifacts": record_image_stats(png_paths),
    "html_artifacts": [rel(path) for path in html_paths],
    "table_artifacts": [rel(path) for path in table_paths],
    "cross_ratio_error": float(cross_ratio_error),
    "numeric_checks": clean_json(check_data),
}

storyboard_path = save_json(clean_json(storyboard), ARTIFACT_ROOT, "checks", "storyboard.json")
visual_checks_path = save_json(clean_json(visual_checks), ARTIFACT_ROOT, "checks", "visual-checks.json")
final_sanity = {
    "chapter": 26,
    "source_span": "printed pp. 505-524 / PDF pp. 527-546",
    "artifacts": [rel(path) for path in all_artifacts],
    "checks": visual_checks,
    "storyboard": rel(storyboard_path),
}
final_sanity_path = save_json(clean_json(final_sanity), ARTIFACT_ROOT, "checks", "final-sanity.json")
assert_artifacts([storyboard_path, visual_checks_path, final_sanity_path], min_size=256)

print(json.dumps({
    "artifact_count": len(all_artifacts),
    "png_count": len(png_paths),
    "html_count": len(html_paths),
    "table_count": len(table_paths),
    "final_sanity": rel(final_sanity_path),
}, indent=2))